# GPM matched timing

This measures fixed, repeated, and adaptive inference on the same GPU, so their times can be compared fairly.

**Before running:**

1. In Google Drive, create a folder called `ada_gpm` directly inside My Drive.
2. Upload `ada_colab_bundle.zip`, `epoch_380.pt`, and `final_evaluation_handoff.zip` into that folder.
3. In Colab, go to **Runtime > Change runtime type**, choose **T4 GPU**, and save.
4. Run the cells below in order, one at a time. Allow Drive access when asked.

If a cell fails, copy its full output and paste it to Claude.


## 1. Connect Drive and unpack the project


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
DRIVE = '/content/drive/MyDrive/ada_gpm'
CHECKPOINT = f'{DRIVE}/epoch_380.pt'
PROJECT = '/content/ada'
HANDOFF = '/content/handoff'
BUNDLE = f'{DRIVE}/ada_colab_bundle.zip'
ARVIND = f'{DRIVE}/final_evaluation_handoff.zip'

for path in (BUNDLE, CHECKPOINT, ARVIND):
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing {path}. Upload it to the ada_gpm folder in Drive.')

zipfile.ZipFile(BUNDLE).extractall(PROJECT)
zipfile.ZipFile(ARVIND).extractall(HANDOFF)
EVAL = f'{PROJECT}/accuracy_efficiency_eval'
print('Project ready.')


## 2. Check the GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 3. Install GPM packages

This takes a few minutes. It must end with `SETUP OK`.


In [ ]:
!python {EVAL}/colab_setup.py


## 4. Run the project tests

This must end with `OK`.


In [ ]:
%cd {EVAL}
!python -m unittest -q test_evaluate


## 5. Measure time

This downloads Computers and runs nine methods in three shuffled rounds. Keep the tab open. Results save to Drive after each round.


In [ ]:
%cd {EVAL}
!python matched_timing.py --gpm-dir {PROJECT}/GPM/GPM --checkpoint {CHECKPOINT} --handoff {HANDOFF} --output {DRIVE}/matched_timing.json


## 6. See the results


In [ ]:
%cd {EVAL}
import json
from IPython.display import Image, display
data = json.load(open(f'{DRIVE}/matched_timing.json'))
print('Device:', data['device'], '| torch:', data['torch'], '| rounds:', data['rounds'])
for r in sorted(data['summary'], key=lambda r: r['ms_per_node']):
    print(f"{r['method']:<19} acc={r['accuracy']:.4f} passes={r['pattern_passes']:6.1f} ms/node={r['ms_per_node']:.4f} speedup={r['speedup_vs_fixed_128']:.2f} ({r['speedup_min']:.2f}-{r['speedup_max']:.2f}) peakMB={r['peak_gpu_mb']:.0f}")
fixed = next(r for r in data['summary'] if r['method'] == 'fixed_128')
print('ACCURACY CHECK PASSED' if abs(fixed['accuracy'] - 0.917) < 0.01 else 'ACCURACY DIFFERS FROM ARVIND RUN: paste this output to Claude.')
!python plot_matched.py --timing {DRIVE}/matched_timing.json --output {DRIVE}/accuracy_vs_time.png
display(Image(f'{DRIVE}/accuracy_vs_time.png'))


## 7. Finish

Copy all the output from cell 6 and paste it to Claude. Results stay in `ada_gpm` in Drive.
